# Enveda v1 submission: FPNet single + 20ppm over 712K precomputed pool
No RDKit at inference. R1 NP-method.

In [ ]:
import sys
print(sys.version)
import torch
print('torch =', torch.__version__, '| cuda =', torch.cuda.is_available())
assert torch.cuda.is_available(), 'need GPU'
import numpy as np, pyarrow.parquet as pq
print('numpy/pyarrow ok')


In [ ]:
from pathlib import Path
def find(name):
    hits = list(Path('/kaggle/input').rglob(name))
    assert hits, 'missing input: ' + name
    print(name, '->', hits[0])
    return str(hits[0])
TEST = find('test.parquet')
W_SINGLE = find('fp_single_s2.pt')
T_FP = find('fp.npy')
T_MASS = find('mass.npy')
T_KEYS = find('keys.npy')
T_SMI = find('smiles.txt')
C_FP = find('coco_fp.npy')
C_MASS = find('coco_mass.npy')
C_META = find('coco_meta.pkl')
print('torch cuda devices:', torch.cuda.device_count())


In [ ]:
import pickle, time
import numpy as np
t0=time.perf_counter()
t_fp = np.load(T_FP, mmap_mode='r')
t_mass = np.load(T_MASS, mmap_mode='r')
t_keys = np.load(T_KEYS, allow_pickle=True)
t_smi = open(T_SMI).read().splitlines()
print('train pool:', t_fp.shape, len(t_keys), len(t_smi), round(time.perf_counter()-t0,1))
assert t_fp.shape[0]==len(t_keys)==len(t_smi)==len(t_mass)==275810
meta = pickle.load(open(C_META,'rb'))
ckeys = list(meta['keys']); csmi = list(meta['smiles'])
c_mass = np.load(C_MASS, mmap_mode='r')
c_fp = np.load(C_FP, mmap_mode='r')
print('coco pool:', c_fp.shape, len(ckeys))
cmap = dict(zip(ckeys, np.asarray(c_mass, float)))
cidx = {k:i for i,k in enumerate(ckeys)}
allk = list(t_keys) + [k for k in ckeys if k not in set(t_keys)]
alls = t_smi + [s for k,s in zip(ckeys,csmi) if k not in set(t_keys)]
allm = np.concatenate([np.asarray(t_mass, float), np.array([cmap[k] for k in ckeys if k not in set(t_keys)], float)])
print('combined:', len(allk), round(time.perf_counter()-t0,1))
TQ = pq.read_table(TEST).to_pydict()
mol = {}
for i in range(len(TQ['molecule_id'])):
    mol.setdefault(TQ['molecule_id'][i], []).append(i)
print('test molecules:', len(mol), 'spectra:', len(TQ['molecule_id']))


In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math
"""Spectrum -> molecular fingerprint model (CSI:FingerID-style neural ranker)."""
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math

MAX_PEAKS = 128
ADDUCT_LIST = ["[M+H]+","[M+NH4]+","[M+Na]+","[M+K]+","[M-H2O+H]+","[M-2H2O+H]+","[M]+",
               "[M-H]-","[M-H2O-H]-","[M+CH2O2-H]-","[M+C2H4O2-H]-","[M+Cl]-","[M]-",
               "[M+2H]2+","[M-2H]-","[2M+H]+","[2M+Na]+","[2M+NH4]+","[2M-H]-","[2M+K]+",
               "[2M+CH2O2-H]-","[2M+C2H4O2-H]-","[2M+Na-2H]-","[M+Na-2H]-","[M-H2O]+","<unk>"]
ADDUCT_IX = {a:i for i,a in enumerate(ADDUCT_LIST)}
INSTR_LIST = ["timsTOF","Orbitrap","QTOF","IT","other"]
INSTR_IX = {a:i for i,a in enumerate(INSTR_LIST)}

def instr_family(s):
    if s is None: return 4
    t = str(s).lower()
    if 'timstof' in t: return 0
    if 'orbitrap' in t or 'qft' in t or 'ftms' in t or 'hybrid ft' in t or 'itft' in t or 'exactive' in t: return 1
    if 'tof' in t: return 2
    if 'trap' in t or 'qq' in t: return 3
    return 4

def prep_peaks(mz, inten, prec_mz, max_peaks=MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):
    """Filter -> window-diversified top-N -> sort by m/z. Returns (mz, sqrt-intensity)."""
    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)
    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    keep = (mz <= prec_mz + 1.5)
    mz, it = mz[keep], it[keep]
    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    mx = it.max()
    if mx <= 0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    keep = it >= floor*mx
    mz, it = mz[keep], it[keep]
    if len(mz) > max_peaks:
        # keep the top `per_win` peaks inside each `win` Da bucket, then global top-N
        order = np.argsort(-it)
        bucket = (mz//win).astype(np.int64)
        cnt = {}; sel=[]
        for i in order:
            b = bucket[i]; c = cnt.get(b,0)
            if c < per_win: cnt[b]=c+1; sel.append(i)
        sel = np.array(sel)
        if len(sel) > max_peaks:
            sel = sel[np.argsort(-it[sel])[:max_peaks]]
        elif len(sel) < max_peaks:
            rest = np.array([i for i in order if i not in set(sel.tolist())])
            need = max_peaks-len(sel)
            if len(rest): sel = np.concatenate([sel, rest[:need]])
        mz, it = mz[sel], it[sel]
    o = np.argsort(mz)
    mz, it = mz[o], it[o]
    v = np.sqrt(it/it.max())
    return mz.astype(np.float32), v.astype(np.float32)

class SinEmb(nn.Module):
    """Log-spaced sinusoidal embedding for m/z values (Voronov et al.)."""
    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):
        super().__init__()
        n = dim//2
        wav = torch.pow(10.0, (hi-lo)*torch.pow(torch.linspace(0,1,n), power) + lo)
        self.register_buffer('inv', (2*math.pi)/wav)
    def forward(self, x):                      # x: (...,)
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], -1)

class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__(); self.h=h
        self.n1=nn.LayerNorm(d); self.qkv=nn.Linear(d,3*d); self.o=nn.Linear(d,d)
        self.n2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d), nn.GELU(), nn.Dropout(drop), nn.Linear(4*d,d))
        self.drop=nn.Dropout(drop)
    def forward(self, x, pad):
        B,N,D=x.shape; y=self.n1(x)
        q,k,v = self.qkv(y).view(B,N,3,self.h,D//self.h).permute(2,0,3,1,4)
        m = (~pad)[:,None,None,:]                       # True = attend
        a = F.scaled_dot_product_attention(q,k,v, attn_mask=m)
        x = x + self.drop(self.o(a.transpose(1,2).reshape(B,N,D)))
        return x + self.drop(self.ff(self.n2(x)))

class FPNet(nn.Module):
    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):
        super().__init__()
        self.d=d
        self.mz_emb  = SinEmb(d)
        self.nl_emb  = SinEmb(d)
        self.pk = nn.Linear(2*d+1, d)
        self.prec_emb = SinEmb(d)
        self.ad = nn.Embedding(len(ADDUCT_LIST), d)
        self.ins = nn.Embedding(len(INSTR_LIST), d)
        self.gl = nn.Linear(d+3, d)
        self.blocks = nn.ModuleList([Block(d,heads,drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2*d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))
    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):
        B,N = mz.shape
        nl = (prec[:,None] - mz).clamp(min=0)
        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))
        g = self.gl(torch.cat([self.prec_emb(prec),
                               (ce/100.0).unsqueeze(-1), mode.unsqueeze(-1),
                               torch.log1p(prec).unsqueeze(-1)/10.0], -1)) + self.ad(ad) + self.ins(ins)
        x = torch.cat([g.unsqueeze(1), p], 1)
        pad = torch.cat([torch.zeros(B,1,dtype=torch.bool,device=pad.device), pad], 1)
        for b in self.blocks: x = b(x, pad)
        x = self.norm(x)
        cls = x[:,0]
        msk = (~pad[:,1:]).float().unsqueeze(-1)
        mean = (x[:,1:]*msk).sum(1)/msk.sum(1).clamp(min=1)
        return self.head(torch.cat([cls, mean], -1))
ck = torch.load(W_SINGLE, map_location='cpu', weights_only=True)
net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).cuda().eval()
net.load_state_dict(ck['model'], strict=True)
print('model ok:', ck['nbits'], ck['d'], ck['layers'])
NTR = 275810
assert t_fp.shape == (275810, 867)


In [ ]:
MASS = dict(C=12.0,H=1.00782503207,N=14.0030740048,O=15.9949146196,P=30.97376163,
    S=31.97207100,F=18.99840322,Cl=34.96885268,Br=78.9183371,I=126.904473,
    Na=22.9897692809,K=38.96370668,Si=27.9769265325,B=11.0093054,Se=79.9165213)
E=0.00054857990; PROTON=MASS['H']-E; H2O=2*MASS['H']+MASS['O']
NH4=MASS['N']+4*MASS['H']; FORMATE=MASS['C']+2*MASS['H']+2*MASS['O']
ACETATE=2*MASS['C']+4*MASS['H']+2*MASS['O']
ADDUCTS = {'[M+H]+':(1,1,PROTON),'[M+NH4]+':(1,1,NH4-E),
 '[M+Na]+':(1,1,MASS['Na']-E),'[M+K]+':(1,1,MASS['K']-E),
 '[M-H2O+H]+':(1,1,PROTON-H2O),'[M-2H2O+H]+':(1,1,PROTON-2*H2O),
 '[M+2H]2+':(1,2,2*PROTON),'[M]+':(1,1,-E),'[M-H2O]+':(1,1,-E-H2O),
 '[M+CH3OH+H]+':(1,1,PROTON+MASS['C']+4*MASS['H']+MASS['O']),
 '[M+CH3CN+H]+':(1,1,PROTON+2*MASS['C']+3*MASS['H']+MASS['N']),
 '[M-H]-':(1,1,-PROTON),'[M-H2O-H]-':(1,1,-PROTON-H2O),
 '[M+CH2O2-H]-':(1,1,FORMATE-PROTON),'[M+C2H4O2-H]-':(1,1,ACETATE-PROTON),
 '[M+Cl]-':(1,1,MASS['Cl']+E),'[M]-':(1,1,E),
 '[M-2H]-':(1,2,-2*PROTON),'[M+Na-2H]-':(1,1,MASS['Na']-2*PROTON),
 '[2M+H]+':(2,1,PROTON),'[2M+Na]+':(2,1,MASS['Na']-E),
 '[2M+NH4]+':(2,1,NH4-E),'[2M+K]+':(2,1,MASS['K']-E),
 '[2M-H]-':(2,1,-PROTON),'[2M+CH2O2-H]-':(2,1,FORMATE-PROTON),
 '[2M+C2H4O2-H]-':(2,1,ACETATE-PROTON),'[2M+Na-2H]-':(2,1,MASS['Na']-2*PROTON),
 '[3M+H]+':(3,1,PROTON),'[3M-H]-':(3,1,-PROTON)}
def neutral_mass(mz, adduct):
    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)
    for a,(n,z,d) in ADDUCTS.items():
        m = (ad == a)
        if m.any(): out[m] = (mz[m]*z - d)/n
    return out
print('mass contract ready')


In [ ]:
import numpy as np, torch, time
@torch.no_grad()
def fwd1(prow):
    a,b = prep_peaks(np.asarray(prow[0],float), np.asarray(prow[1],float), float(prow[2]))
    if not len(a): return None
    N = len(a)
    mz=np.zeros((1,N),np.float32); it=np.zeros((1,N),np.float32); pad=np.zeros((1,N),bool)
    mz[0,:N]=a; it[0,:N]=b
    T = lambda x: torch.as_tensor(x, device='cuda')
    args=(T(mz),T(it),T(pad),T(np.array([float(prow[2])],np.float32)),
          T(np.array([ADDUCT_IX.get(prow[3],ADDUCT_IX['<unk>'])])),
          T(np.array([instr_family(prow[4])])),
          T(np.array([float(np.mean(np.atleast_1d(prow[5]))) if prow[5] is not None else 25.0],np.float32)),
          T(np.array([1.0 if prow[6]=='positive' else -1.0],np.float32)))
    return net(*args).float().cpu().numpy()[0]
def pool_fp_row(j):
    if j < NTR:
        return np.unpackbits(np.asarray(t_fp[j], dtype=np.uint8))[:6930].astype(np.float32)
    return np.unpackbits(np.asarray(c_fp[cidx[allk[j]]], dtype=np.uint8))[:6930].astype(np.float32)
out_rows = []
stat = {'empty_window':0,'mass_fallback':0,'cco':0,'exception':0}
t0=time.perf_counter()
for n,(mid,idxs) in enumerate(mol.items()):
    try:
        best = max(idxs, key=lambda i: len(TQ['ms2_mzs'][i]))
        prow = (TQ['ms2_mzs'][best], TQ['ms2_normalized_intensities'][best], TQ['precursor_mz'][best],
                TQ['adduct'][best], TQ['instrument_type'][best], TQ['collision_energy_ev'][best], TQ['ionization_mode'][best])
        z = fwd1(prow)
        qm = neutral_mass(np.array([prow[2]]), [prow[3]])[0]
        cands = []
        if not np.isnan(qm):
            w = np.where(np.abs(allm - qm) <= qm*20e-6)[0]
            if len(w) > 2000:
                w = w[np.argsort(np.abs(allm[w]-qm))[:2000]]
            is_tr = w < NTR
            Fmat = np.empty((len(w), 6930), np.float32)
            if is_tr.any():
                Fmat[is_tr] = np.unpackbits(np.asarray(t_fp[w[is_tr]], dtype=np.uint8), axis=1)[:, :6930]
            if (~is_tr).any():
                ci = np.array([cidx[allk[j]] for j in w[~is_tr]])
                Fmat[~is_tr] = np.unpackbits(np.asarray(c_fp[ci], dtype=np.uint8), axis=1)[:, :6930]
            F = [(Fmat[i], allk[j], alls[j], abs(allm[j]-qm)/qm*1e6) for i, j in enumerate(w)]
            if not F:
                stat['empty_window'] += 1
            if z is not None and F:
                Fall = np.stack([f.astype(np.float32) for f,_,_,_ in F])
                s = Fall @ z
                order = np.argsort(-s)
                seen_k, cands = set(), []
                for j in order:
                    _,k,sm,_ = F[j]
                    if k in seen_k: continue
                    seen_k.add(k); cands.append(sm)
                    if len(cands) >= 25: break
            if not cands and F:
                stat['mass_fallback'] += 1
                fb = sorted(F, key=lambda t: t[3])[:25]
                seen_k, cands = set(), []
                for _,k,sm,_ in fb:
                    if k in seen_k: continue
                    seen_k.add(k); cands.append(sm)
        if not cands:
            cands = ['CCO']*25
            stat['cco'] += 1
        else:
            cands = (cands + ['CCO']*25)[:25]
        out_rows.append((mid, ';'.join(cands)))
    except Exception as e:
        stat['exception'] += 1
        out_rows.append((mid, ';'.join(['CCO']*25)))
    if (n+1)%100==0: print(n+1, round(time.perf_counter()-t0,1), stat)
import csv
with open('/kaggle/working/submission.csv','w',newline='') as f:
    w = csv.writer(f); w.writerow(['molecule_id','smiles'])
    w.writerows(out_rows)
print('wrote', len(out_rows), round(time.perf_counter()-t0,1))
print('coverage:', stat)
assert len(out_rows)==len(mol)


产物 `/kaggle/working/submission.csv`。覆盖率随行打印；CCO 仅空窗/异常时出现。